In [ ]:
# Cell 01 | Verify the Python runtime
# Purpose: Confirm the Python version and executable used by the current Jupyter kernel.
# Key points: 理解 Notebook kernel 實際使用哪個 Python 環境，避免套件已安裝但執行時找不到的環境錯置問題。
# Execution: 可獨立執行；輸出 Python version 與 executable path，確認目前使用專案預期的虛擬環境。

import sys

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

Python version:
3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]



Python executable:
g:\01.Project\rag-knowledge-assistant\.venv\Scripts\python.exe


In [ ]:
# Cell 02 | Load environment variables and initialize the OpenAI client
# Purpose: Load local API credentials and create the OpenAI client used for LLM requests.
# Key points: .env 將 secrets 與程式碼分離；OpenAI SDK 會從環境變數讀取 OPENAI_API_KEY，避免將金鑰寫入 Notebook。
# Execution: 需確認專案根目錄的 .env 已設定；成功後 client 可供後續模型呼叫使用。

from dotenv import load_dotenv
from openai import OpenAI

#讀取專案根目錄中的 .env
load_dotenv("../.env")

#OpenAI SDK 會自動讀取 OPENAI_API_KEY
client = OpenAI()

print("OpenAI client initialized successfully.")

OpenAI client initialized successfully.


In [ ]:
# Cell 03 | Test the LLM connection
# Purpose: Verify that the OpenAI client, API credential, and selected model can complete a basic request.
# Key points: 先獨立驗證 Generation 能正常運作，後續才能區分問題來自 retrieval、prompt construction 或 LLM API。
# Execution: 需先完成 Cell 02；成功後應取得模型產生的簡短文字答案。

response = client.responses.create(
    model="gpt-5.6",
    input="Explanin RAG in one short sentence for a beginner."
)

print(response.output_text)


RAG helps AI give better answers by looking up relevant information before responding.


In [ ]:
# Cell 04 | Load the course catalog
# Purpose: Retrieve the DataTalks.Club course catalog used to discover the available FAQ data sources.
# Key points: courses.json 提供各課程的 metadata 與 FAQ path；這一步先取得資料來源清單，尚未載入完整 FAQ documents。
# Execution: 需要網路連線；成功後 courses_raw 應包含可供下一格逐一下載 FAQ 的課程資訊。

import requests

docs_url = "https://datatalks.club/faq/json/courses.json"

response = requests.get(docs_url)
response.raise_for_status()

courses_raw = response.json()

print("Number of courses:", len(courses_raw))
print("\nFirst course:")
print(courses_raw[0])

Number of courses: 6

First course:
{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}


In [ ]:
# Cell 05 | Build the FAQ document collection
# Purpose: Download FAQ entries for each course and combine them into one document collection for retrieval.
# Key points: 將 course catalog 中的不同 FAQ source 整合為統一的 documents list；每筆 document 保留 course、section、question、answer 等欄位。
# Execution: 需先完成 Cell 04；成功後 documents 應包含完整 FAQ collection，並輸出總筆數與第一筆資料供結構確認。

documents = []

url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f'{url_prefix}{course["path"]}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()

    course_data = course_response.json()

    documents.extend(course_data)

print("Number of documents:", len(documents))

print("\nFirst document:")
print(documents[0])

Number of documents: 1401

First document:
{'id': '9e508f2212', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: When does the course start?', 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}


In [ ]:
# Cell 06 | Build the MinSearch index
# Purpose: Index the FAQ documents so relevant entries can be retrieved from a natural-language question.
# Key points: text_fields 參與文字相關性搜尋，keyword_fields 用於精確篩選；index.fit() 將 documents 建立成可查詢索引。
# Execution: 需先完成 Cell 05；成功後 index 可供後續 retrieval 使用。

from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

print("Search index created successfully.")

Search index created successfully.


In [ ]:
# Cell 07 | Retrieve relevant FAQ documents
# Purpose: Search the MinSearch index for FAQ entries that are most relevant to the user's question.
# Key points: boost_dict 控制不同文字欄位的搜尋權重，filter_dict 限定課程範圍，num_results 控制回傳的候選文件數量。
# Execution: 需先完成 Cell 06；成功後 search_results 應包含依相關性排序的 FAQ documents。

question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={
        "question": 2.0,
        "section": 0.5
    },
    filter_dict={
        "course": "llm-zoomcamp"
    },
    num_results=5
)

for i, doc in enumerate(search_results, start=1):
    print(f"{i}. {doc['question']}")

1. I just discovered the course. Can I still join?
2. Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
3. Why do we need orchestration / Kestra — can't I just run the code in a notebook?
4. Certificate: Can I follow the course in a self-paced mode and get a certificate?
5. How should I start the course and follow the weekly workflow?


In [ ]:
# Cell 08 | Inspect the top retrieval result
# Purpose: Examine the highest-ranked FAQ document before using retrieved content in the prompt.
# Key points: Retrieval quality directly affects RAG answer quality；先確認 Top-1 result 是否真的包含相關資訊，再進入 prompt construction。
# Execution: 需先完成 Cell 07；執行後檢查最佳結果的 course、section、question、answer 是否符合使用者問題。

best_match = search_results[0]

print("Course:")
print(best_match["course"])

print("\nSection:")
print(best_match["section"])

print("\nQuestion:")
print(best_match["question"])

print("\nAnswer:")
print(best_match["answer"])

Course:
llm-zoomcamp

Section:
General Course-Related Questions

Question:
I just discovered the course. Can I still join?

Answer:
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [ ]:
# Cell 09 | Define prompt templates
# Purpose: Separate fixed LLM instructions from the dynamic question and retrieved context used for each request.
# Key points: INSTRUCTIONS 定義模型行為與 grounding 規則；USER_PROMPT_TEMPLATE 定義每次 request 中 question 與 context 的輸入結構。
# Execution: 本格只建立 reusable prompt components；下一步會將 search_results 整理成 context，再填入 user prompt template。

INSTRUCTIONS = """
You answer questions from course participants using only the provided context.

Use the context to provide an accurate answer.
If the context does not contain the answer, say that you don't know.
""".strip()

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
""".strip()


print("=== INSTRUCTIONS ===")
print(INSTRUCTIONS)

print("\n=== USER PROMPT TEMPLATE ===")
print(USER_PROMPT_TEMPLATE)

=== INSTRUCTIONS ===
You answer questions from course participants using only the provided context.

Use the context to provide an accurate anser.
If the context does not contain the answer, say that you don't know.

=== USER PROMPT TEMPLATE ===
Question:
{question}

Context:
{context}


In [ ]:
# Cell 10 | Build retrieved context
# Purpose: Convert MinSearch results into a prompt-ready context string for the LLM.
# Key points: 理解 list[dict] → string 的資料轉換，並保留 section、question、answer，讓 retrieval results 能成為模型可閱讀的 context。
# Execution: 需先完成 Cell 07 取得 search_results；輸出的 context 會在下一步與 question 組成完整 User Prompt。

def build_context(search_results):
    """Convert retrieved FAQ documents into a prompt-ready context string."""
    lines = []

    # Format each retrieved document as a readable FAQ block.
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    # Join all FAQ blocks into one context string for prompt construction.
    return "\n".join(lines).strip()


context = build_context(search_results)

print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchestrator like Kestra provides.

In this module Kestra is also the veh

In [13]:
# Cell 11 | Build the user prompt
# Purpose: Combine the user's question with the retrieved context using the reusable prompt template.
# Key points: 將動態的 question 與 context 填入 USER_PROMPT_TEMPLATE，完成 Retrieval → Prompt 的資料銜接。
# Execution: 需先完成 Cell 09 與 Cell 10；輸出的 user_prompt 會在下一步傳給 LLM 產生 grounded answer。

def build_prompt(question, search_results):
    """Build a user prompt from the question and retrieved FAQ documents."""
    context = build_context(search_results)

    user_prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context,
    )

    return user_prompt.strip()


user_prompt = build_prompt(question, search_results)

print(user_prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchest

In [14]:
# Cell 12 | Generate a grounded answer
# Purpose: Send the fixed instructions and retrieval-grounded user prompt to the LLM and generate an answer.
# Key points: developer message 負責固定行為規則，user message 傳入 question + retrieved context；這一步完成 Prompt → Generation。
# Execution: 需先完成 Cell 09～11；成功後應取得一個主要根據 FAQ context 產生的 answer。

message_history = [
    {
        "role": "developer",
        "content": INSTRUCTIONS,
    },
    {
        "role": "user",
        "content": user_prompt,
    },
]

response = client.responses.create(
    model="gpt-5.6",
    input=message_history,
)

answer = response.output_text

print(answer)

Yes, you can join and start learning now. If you want a certificate, you must submit your capstone project and complete the required peer reviews while the live cohort is still accepting submissions.


In [15]:
# Cell 13 | Wrap the LLM call
# Purpose: Encapsulate the OpenAI Responses API call into a reusable function for the RAG pipeline.
# Key points: 將 model request 與 prompt construction 分離；llm() 只負責接收 instructions、user_prompt 並回傳模型文字結果。
# Execution: 需先完成 Cell 02 建立 client；執行後會用現有 prompt 測試 reusable LLM function。

def llm(instructions, user_prompt, model="gpt-5.6"):
    """Generate an LLM response from fixed instructions and a dynamic user prompt."""
    message_history = [
        {
            "role": "developer",
            "content": instructions,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    response = client.responses.create(
        model=model,
        input=message_history,
    )

    return response.output_text


answer = llm(INSTRUCTIONS, user_prompt)

print(answer)

Yes, you can join and start learning now. If you want a certificate, you must submit a capstone project and complete the required peer reviews while the live cohort is still accepting submissions.


In [17]:
# Cell 14 | Wrap the retrieval logic
# Purpose: Encapsulate MinSearch retrieval into a reusable search() function for the RAG pipeline.
# Key points: 將 Cell 07 已驗證的 index.search() 邏輯封裝；search() 只負責接收 query 並回傳相關 FAQ documents。
# Execution: 需先完成 Cell 06 建立 index；執行後會用現有問題測試 reusable retrieval function。

def search(query, course="llm-zoomcamp"):
    """Retrieve the most relevant FAQ documents for a query."""
    boost_dict = {
        "question": 2.0,
        "section": 0.5,
    }

    filter_dict = {
        "course": course,
    }

    return index.search(
        query=query,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5,
    )


search_results = search(
    "I just discovered the course. Can I join now?"
)

print(search_results[0])

{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}


In [ ]:
# Cell 15 | Build the complete RAG pipeline
# Purpose: Connect retrieval, prompt construction, and LLM generation into one reusable end-to-end RAG function.
# Key points: rag() 負責協調整體流程；search() 找資料、build_prompt() 建立 grounded prompt、llm() 根據 context 產生答案。
# Execution: 需先完成 search()、build_prompt() 與 llm()；輸入一個 query 後即可執行完整 RAG pipeline。

def rag(query, model="gpt-5.6"):
    """Run the end-to-end retrieval-augmented generation pipeline."""
    search_results = search(query)
    user_prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, user_prompt, model=model)

    return answer


answer = rag("I just discovered the course. Can I join now?")

print(answer)

Yes, you can join and start learning now. If you want a certificate, you must submit a capstone project and complete the required peer reviews while the live cohort is still accepting submissions.


In [19]:
# Cell 16 | Test RAG with a new question
# Purpose: Verify that the end-to-end RAG pipeline can answer a different question using the same reusable components.
# Key points: 測試 pipeline 是否具有可重用性，而不是只對前面反覆使用的單一問題有效；答案應主要根據 retrieved FAQ context。
# Execution: 需先完成 Cell 15 建立 rag()；本格會送出一個新的 course-related query 並檢查最終答案。

test_query = "How do I get a certificate?"

answer = rag(test_query)

print(answer)

To get a certificate, you must:

- Complete and pass the capstone project.
- Complete the required peer reviews.
- Submit the project and peer reviews while a live cohort is accepting them.

Homework is not required, though it is recommended. You may study and prepare your project in self-paced mode, but certificates are only available through a live cohort.


In [20]:
# Cell 17 | Trace the retrieved evidence
# Purpose: Inspect the documents retrieved for a RAG answer and verify which FAQ entries support the generated response.
# Key points: RAG debugging 要區分 Retrieval 與 Generation；先確認 search() 找到正確 evidence，再判斷 LLM 是否忠實使用 context。
# Execution: 使用 Cell 16 的 test_query 重新執行 retrieval；輸出 Top-5 results 的排名、section、question 與 answer。

retrieved_docs = search(test_query)

for rank, doc in enumerate(retrieved_docs, start=1):
    print(f"Rank {rank}")
    print(f"Section: {doc['section']}")
    print(f"Q: {doc['question']}")
    print(f"A: {doc['answer']}")
    print()

Rank 1
Section: General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

To get the certificate, you need to finish a capstone project and complete the
required peer reviews. Homework is not required. You can work through the
material and prepare your project in self-paced mode, but project submission and
peer review must happen while a live cohort is accepting them.

Rank 2
Section: General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

Rank 3
Section: Module 1 Homework
Q: How do I get token counts for Module 1 homework if I use a different provider?
A: For the current Module 1 homework, 

In [21]:
# Cell 18 | Test the unknown-answer behavior
# Purpose: Verify that the RAG pipeline avoids inventing an answer when the retrieved context does not contain the requested information.
# Key points: Grounding 不只要求「有資料時答對」，也要求「沒有 evidence 時不要依賴模型自身知識亂補答案」。
# Execution: 使用一個超出課程 FAQ 範圍的問題；理想結果應明確表示無法從提供的 context 得知答案。

unknown_query = "What is the current weather in Taipei?"

answer = rag(unknown_query)

print(answer)

I don’t know. The provided context does not include current weather information for Taipei.


In [22]:
# Cell 19 | Test the ingestion module
# Purpose: Verify that FAQ loading and MinSearch index creation work correctly after moving them from the Notebook into ingest.py.
# Key points: 驗證 module import、data ingestion 與 index construction 三個環節；重構後的功能應與原本 Cell 04～06 行為一致。
# Execution: 需先建立 ingest.py；執行後應成功載入 documents、建立 index，並用一個測試問題取得相關 FAQ。

from ingest import load_faq_data, build_index


test_documents = load_faq_data()
test_index = build_index(test_documents)

test_results = test_index.search(
    query="I just discovered the course. Can I join now?",
    boost_dict={
        "question": 2.0,
        "section": 0.5,
    },
    filter_dict={
        "course": "llm-zoomcamp",
    },
    num_results=1,
)

print("Documents loaded:", len(test_documents))
print("Top result:")
print(test_results[0])

Documents loaded: 1401
Top result:
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}


In [26]:
# Cell 20 | Test the RAG helper
# Purpose: Verify that the refactored RAGBase class can run the same end-to-end RAG workflow as the Notebook prototype.
# Key points: 將既有 index 與 OpenAI client 注入 RAGBase；驗證 class 能獨立協調 retrieval、prompt construction 與 generation。
# Execution: 需先完成 Cell 02 建立 client、Cell 19 建立 test_index，以及建立 rag_helper.py；執行後應取得 grounded answer。

from rag_helper import RAGBase


assistant = RAGBase(
    index=test_index,
    llm_client=client,
)

test_query = "I just discovered the course. Can I join now?"

answer = assistant.rag(test_query)

print(answer)

Yes, you can join and start learning at any time. If you want a certificate, you must submit a capstone project and complete the required peer reviews while a live cohort is still accepting submissions.


In [27]:
# Cell 21 | Verify grounding after refactoring
# Purpose: Confirm that RAGBase preserves the unknown-answer behavior after moving the RAG logic out of the Notebook.
# Key points: Refactoring 應只改變程式結構，不改變 grounding 行為；當 retrieved context 不含答案時，模型應明確表示不知道。
# Execution: 需先完成 Cell 20 建立 assistant；執行後應拒絕根據模型自身知識回答與 FAQ 無關的問題。

unknown_query = "What is the current weather in Taipei?"

answer = assistant.rag(unknown_query)

print(answer)

I don’t know. The provided context does not include current weather information for Taipei.
